# Notebook 3 — Advanced Prompting Strategies

**Topics covered in this notebook:**
7. Prompt Chaining and Decomposition Strategies
8. Meta-Prompting and Self-Refinement Loops

---

## ⚙️ Setup — Pick your API provider (free options available!)

| Provider | Cost | Where to get a key |
|----------|------|--------------------|
| **Groq** ✅ FREE | No credit card | [console.groq.com/keys](https://console.groq.com/keys) |
| **Gemini** ✅ FREE | No credit card | [aistudio.google.com/app/apikey](https://aistudio.google.com/app/apikey) |
| **OpenAI** | Paid | [platform.openai.com/api-keys](https://platform.openai.com/api-keys) |

> These techniques power real production AI systems — pipelines, agents, and automated feedback loops.
> By the end you'll be able to build multi-step reasoning systems from scratch.

In [1]:
# ── CELL 0 · Install dependencies (run once) ─────────────────────────────────
import subprocess
result = subprocess.run(
    ["uv", "pip", "install",
     "openai>=3.8.0",
     "pydantic>=2.12.0",
     "python-dotenv>=1.0.0",
     "rich>=14.0.0"],
    capture_output=True, text=True
)
print(result.stdout or "All packages already installed.")

All packages already installed.


In [2]:
# ── CELL 1 · Provider auto-detection & client setup ─────────────────────────
import os
import json
from typing import Optional
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field
from rich import print as rprint
from rich.panel import Panel
from rich.rule import Rule
from rich.text import Text

load_dotenv()

# ── Provider detection — priority: OpenAI > Groq > Gemini ────────────────────
OPENAI_KEY  = os.getenv("OPENAI_API_KEY", "")
GROQ_KEY    = os.getenv("GROQ_API_KEY", "")
GEMINI_KEY  = os.getenv("GEMINI_API_KEY", "")

if OPENAI_KEY and not OPENAI_KEY.startswith("sk-..."):
    PROVIDER = "openai"
    client = OpenAI(api_key=OPENAI_KEY)
    MODEL = "gpt-4o"
elif GROQ_KEY and not GROQ_KEY.startswith("gsk_..."):
    PROVIDER = "groq"
    client = OpenAI(
        api_key=GROQ_KEY,
        base_url="https://api.groq.com/openai/v1",
    )
    MODEL = "openai/gpt-oss-120b"
elif GEMINI_KEY and not GEMINI_KEY.startswith("AIza..."):
    PROVIDER = "gemini"
    client = OpenAI(
        api_key=GEMINI_KEY,
        base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    )
    MODEL = "models/gemini-3.7-flash"
else:
    raise EnvironmentError(
        "No valid API key found in .env!\n"
        "Add one of: OPENAI_API_KEY, GROQ_API_KEY, or GEMINI_API_KEY\n"
        "See .env.example for instructions."
    )

print(f"✓ Provider : {PROVIDER.upper()}")
print(f"✓ Model    : {MODEL}\n")

# ── Shared helper functions ───────────────────────────────────────────────────
def chat(messages: list[dict], model: str = MODEL, temperature: float = 0.3,
         response_format: dict | None = None) -> str:
    kwargs = dict(model=model, messages=messages, temperature=temperature)
    if response_format:
        kwargs["response_format"] = response_format
    response = client.chat.completions.create(**kwargs)
    return response.choices[0].message.content

def show(content: str, title: str, border: str = "green") -> None:
    rprint(Panel(str(content), title=f"[bold]{title}[/]", border_style=border))

def step_banner(n: int, title: str) -> None:
    rprint(Rule(f"[bold cyan]Step {n}: {title}[/]"))

✓ Provider : GROQ
✓ Model    : openai/gpt-oss-120b



---
## Part 7 — Prompt Chaining and Decomposition Strategies

**Prompt chaining** = breaking a complex task into a sequence of smaller, simpler prompts  
where the **output of one prompt becomes the input to the next**.

### Why chain instead of one mega-prompt?
- Each step can have a **specialized instruction** tailored to that sub-task
- You can **inspect and validate** each intermediate result
- Errors stay contained — easier to debug which step failed
- Allows **conditional branching** (route to different chains based on step output)
- Enables **parallel execution** of independent sub-tasks

### Chain patterns
| Pattern | Description | Use when |
|---------|------------|----------|
| **Sequential** | A → B → C (linear pipeline) | Ordered transformation |
| **Map-Reduce** | Process N items → aggregate | Summarizing many documents |
| **Parallel** | A + B (independent, then merge) | Faster multi-aspect analysis |
| **Gated/Conditional** | Route based on step output | Classification → specialized handler |

### Research insight
> Prompt chaining improves accuracy by **+10–30%** on complex multi-step tasks vs a single mega-prompt. *(SurePrompts, 2026)*  
> Each individual prompt in a chain should stay focused on **one clear task**.

In [3]:
# ── EXAMPLE 7a · Sequential chain — Document Analysis Pipeline ───────────────
# Pipeline: raw document → extract facts → verify facts → generate report

raw_document = """\
INTERNAL MEMO — Q3 2024 Performance Review

Revenue came in at $42.3M, up 18% YoY. However, CAC (Customer Acquisition Cost) 
increased 34% to $1,847 per customer. Churn rose to 6.2% from 4.1% last quarter. 
The new enterprise tier launched in July contributed $8.1M — ahead of the $6M forecast.
Engineering headcount grew from 47 to 63 (34% increase), driving OpEx up 28%.
Net margin compressed from 22% to 14% despite top-line growth.
"""

# ── Step 1: Extract key metrics ─────────────────────────────────────────────
step_banner(1, "Extract Key Metrics")
step1_prompt = f"""\
Extract all numerical metrics from this business memo.
Return a JSON object where each key is a metric name (snake_case) 
and each value is the number (as a float or percentage string).
Include YoY/QoQ comparisons as separate keys with suffix _prev.

Memo:
{raw_document}
"""
metrics_raw = chat([{"role": "user", "content": step1_prompt}],
                   response_format={"type": "json_object"}, temperature=0.0)
metrics = json.loads(metrics_raw)
rprint(metrics)

# ── Step 2: Identify concerning trends ──────────────────────────────────────
step_banner(2, "Identify Concerning Trends")
step2_prompt = f"""\
You are a CFO analyzing business metrics. Given these Q3 metrics:
{json.dumps(metrics, indent=2)}

Identify the top 3 most concerning trends that leadership should address urgently.
For each concern:
- Name the metric
- Explain WHY it's concerning (business impact)
- Suggest ONE specific investigation to run

Format: numbered list, concise.
"""
concerns = chat([{"role": "user", "content": step2_prompt}], temperature=0.2)
show(concerns, "Top Concerns Identified", "yellow")

# ── Step 3: Generate executive summary ──────────────────────────────────────
step_banner(3, "Generate Executive Summary")
step3_prompt = f"""\
Write a concise executive summary for the board.

Raw data:
{raw_document}

Key concerns flagged by CFO analysis:
{concerns}

Format:
- Headline (1 sentence with the #1 story)
- Positives (2 bullets)
- Risks (2 bullets, from the concerns above)
- Recommended board action (1 sentence)

Tone: Board-level, direct, no jargon.
"""
summary = chat([{"role": "user", "content": step3_prompt}], temperature=0.2)
show(summary, "Final Executive Summary (3-step chain)", "green")

─────────────────────────────────────────── Step 1: Extract Key Metrics ───────────────────────────────────────────

{
    'revenue': 42.3,
    'revenue_yoy': '18%',
    'cac': 1847,
    'cac_change': '34%',
    'churn': '6.2%',
    'churn_prev': '4.1%',
    'enterprise_tier_revenue': 8.1,
    'enterprise_tier_forecast': 6.0,
    'engineering_headcount': 63,
    'engineering_headcount_prev': 47,
    'engineering_headcount_change': '34%',
    'opex_change': '28%',
    'net_margin': '14%',
    'net_margin_prev': '22%'
}

─────────────────────────────────────── Step 2: Identify Concerning Trends ────────────────────────────────────────

╭──────────────────────────────────────────── Top Concerns Identified ────────────────────────────────────────────╮
│ 1. **Customer Acquisition Cost (CAC) – +34% YoY**                                                               │
│    - **Why it matters:** A sharply rising CAC erodes unit economics and threatens sustainable growth; if        │
│ acquisition costs outpace revenue gains, cash burn will accelerate.                                             │
│    - **Investigation:** Run a **cohort‑level CAC attribution analysis** to pinpoint which channels, campaigns,  │
│ or customer segments are driving the cost surge and compare their conversion efficiency to historical           │
│ benchmarks.                                                                                                     │
│                                                                                                                 │
│ 2. **Churn Rate – 6.2% (up from 4.1%)**                                                                         │
│    - **Why it matters:** Higher churn reduces the lifetime value of each customer, offsets new‑sale revenue,    │
│ and signals possible product, service, or pricing issues that can stall scaling.                                │
│    - **Investigation:** Conduct a **customer‑exit survey and usage‑behavior segmentation** to identify common   │
│ pain points or usage patterns among churned accounts and quantify their impact on revenue loss.                 │
│                                                                                                                 │
│ 3. **Net Margin – 14% (down from 22%)**                                                                         │
│    - **Why it matters:** A 8‑point margin contraction indicates that operating expenses are rising faster than  │
│ revenue, threatening profitability and limiting cash available for strategic investments.                       │
│    - **Investigation:** Perform a **detailed OPEX variance drill‑down** (by department, cost category, and      │
│ headcount) to isolate the primary drivers of the 28% OPEX increase and assess cost‑saving opportunities.        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

─────────────────────────────────────── Step 3: Generate Executive Summary ────────────────────────────────────────

╭──────────────────────────────────── Final Executive Summary (3-step chain) ─────────────────────────────────────╮
│ **Headline:** Q3 revenue surged 18% to $42.3 M, but rising CAC, higher churn, and a compressed net margin       │
│ signal emerging profitability pressure.                                                                         │
│                                                                                                                 │
│ **Positives**                                                                                                   │
│ - Enterprise tier launch delivered $8.1 M, beating the $6 M forecast and expanding our high‑margin product mix. │
│ - Overall top‑line growth outpaced the prior year, confirming market demand for our core offering.              │
│                                                                                                                 │
│ **Risks**                                                                                                       │
│ - CAC jumped 34% YoY to $1,847, threatening unit economics and cash burn if acquisition efficiency does not     │
│ improve.                                                                                                        │
│ - Churn climbed to 6.2% (from 4.1%), eroding customer lifetime value and indicating possible product or service │
│ gaps.                                                                                                           │
│                                                                                                                 │
│ **Recommended board action:** Approve a focused cost‑efficiency task force to dissect CAC drivers and churn     │
│ causes, and to identify OPEX reductions that can restore margin targets.                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [4]:
# ── EXAMPLE 7b · Map-Reduce chain — Summarize multiple documents ─────────────
# Map: summarize each document independently
# Reduce: synthesize all summaries into one final output

# Simulating 4 customer reviews to aggregate
reviews = [
    """Review 1: Setup took over an hour — the instructions are terrible. 
    Once running though, it's fast and reliable. WiFi range is impressive.""",
    
    """Review 2: Plug and play, worked in 10 minutes. The app is intuitive. 
    However, it disconnects every few days and needs a manual restart.""",
    
    """Review 3: Best router I've owned. Handles 30+ devices without slowdown. 
    The parental controls are excellent. Price is high but worth it.""",
    
    """Review 4: Returned it after a week. Customer support was unhelpful 
    when I had connectivity issues. Build quality feels cheap for the price.""",
]

# ── MAP phase: summarize each review independently ───────────────────────────
print("MAP phase — summarizing each review...")
map_template = """\
Summarize this product review in exactly 3 key points:
1. Main positive (if any)
2. Main negative (if any)  
3. Overall verdict in 5 words

Review: {review}
"""

summaries = []
for i, review in enumerate(reviews, 1):
    summary = chat([{"role": "user", "content": map_template.format(review=review)}], temperature=0.0)
    summaries.append(summary)
    print(f"  ✓ Review {i} summarized")

# ── REDUCE phase: synthesize all summaries ───────────────────────────────────
print("\nREDUCE phase — synthesizing all summaries...")
all_summaries = "\n\n".join(f"[Review {i+1}]\n{s}" for i, s in enumerate(summaries))

reduce_prompt = f"""\
You've received summaries of 4 customer reviews for a WiFi router.
Synthesize them into a balanced product assessment.

Individual summaries:
{all_summaries}

Write:
1. Overall rating (1–5 stars) with one-sentence justification
2. Top 3 strengths (mentioned by multiple reviewers)
3. Top 3 weaknesses (mentioned by multiple reviewers)
4. Who should buy this? (1 sentence)
5. Who should avoid this? (1 sentence)
"""

final_assessment = chat([{"role": "user", "content": reduce_prompt}], temperature=0.2)
show(final_assessment, "Map-Reduce: Aggregated Product Assessment", "green")

MAP phase — summarizing each review...
  ✓ Review 1 summarized
  ✓ Review 2 summarized
  ✓ Review 3 summarized
  ✓ Review 4 summarized

REDUCE phase — synthesizing all summaries...


╭─────────────────────────────────── Map-Reduce: Aggregated Product Assessment ───────────────────────────────────╮
│ **Overall rating:** ★★★☆☆ – The router delivers strong, wide‑area performance, but recurring connectivity       │
│ glitches, pricey positioning and sub‑par documentation keep it from being a clear winner.                       │
│                                                                                                                 │
│ **Top 3 strengths (cited by multiple reviewers)**                                                               │
│ 1. **Robust Wi‑Fi performance and range** – reviewers praised fast, reliable speeds and a large coverage area.  │
│ 2. **Capacity for many devices** – it can comfortably support 30 + simultaneous connections without noticeable  │
│ slowdown.                                                                                                       │
│ 3. **Convenient initial setup (plug‑and‑play/app‑driven)** – at least one user found the out‑of‑the‑box         │
│ experience very straightforward.                                                                                │
│                                                                                                                 │
│ **Top 3 weaknesses (cited by multiple reviewers)**                                                              │
│ 1. **Intermittent connectivity** – several users reported frequent drops that required manual restarts.         │
│ 2. **Poor setup documentation** – the instructions are confusing or missing, making the initial configuration   │
│ frustrating for some.                                                                                           │
│ 3. **High price point** – the cost is above average for comparable routers, which may deter budget‑focused      │
│ buyers.                                                                                                         │
│                                                                                                                 │
│ **Who should buy this?**                                                                                        │
│ Anyone who needs a high‑capacity, long‑range Wi‑Fi solution for a busy household or small office and is willing │
│ to pay a premium for performance.                                                                               │
│                                                                                                                 │
│ **Who should avoid this?**                                                                                      │
│ Users who prioritize rock‑solid reliability, low cost, or responsive customer support should look elsewhere.    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [5]:
# ── EXAMPLE 7c · Gated/Conditional chain — Support Ticket Router ─────────────
# Step 1: Classify ticket category
# Step 2: Route to the appropriate specialized handler based on category
# This is how real support automation pipelines work.

# Specialized response handlers for each category
HANDLERS = {
    "Billing": """\
You are a billing specialist. Be empathetic and precise.
Always mention: our billing cycle, how to download invoices, and escalation path to finance team.
Offer a specific resolution, not generic "we'll look into it".""",

    "Technical Bug": """\
You are a Level 2 technical support engineer. Be systematic.
Always ask for: browser/OS version, steps to reproduce, error messages.
Provide immediate workarounds if available, then log the bug for engineering.""",

    "Feature Request": """\
You are a product manager. Be appreciative and transparent.
Acknowledge the request, explain our prioritization process, and ask for use case details.
Never promise a timeline. Do add to the feedback database.""",

    "Account Access": """\
You are a security-focused account specialist.
Prioritize account security above all. Follow our verification protocol before any changes.
Provide self-service options (password reset link, 2FA setup) before manual intervention.""",
}

def route_and_respond(ticket: str) -> tuple[str, str]:
    """Classify a support ticket then generate a specialized response."""
    
    # Gate: classify first
    classify_prompt = f"""\
Classify this support ticket into exactly one category.
Choose from: Billing, Technical Bug, Feature Request, Account Access, Other
Return only the category name, nothing else.

Ticket: "{ticket}"
Category:"""
    category = chat([{"role": "user", "content": classify_prompt}], temperature=0.0).strip()
    
    # Route to specialized handler (or fallback to generic)
    system_prompt = HANDLERS.get(category, "You are a helpful customer support agent.")
    
    response = chat([
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Support ticket: {ticket}"}
    ], temperature=0.3)
    
    return category, response


# Test with 3 different tickets
tickets = [
    "I was charged twice last month and the charge appeared on my card but not in my dashboard.",
    "The CSV export button does nothing when I click it in Chrome. I'm on macOS 14.",
    "Would be amazing if you added dark mode! It would make night-time usage much easier.",
]

for ticket in tickets:
    category, response = route_and_respond(ticket)
    rprint(Rule(f"[bold]Ticket:[/] {ticket[:60]}..."))
    rprint(f"[bold yellow]→ Routed to:[/] {category}")
    show(response, f"{category} Handler Response", "blue")
    print()

───────────────────── Ticket: I was charged twice last month and the charge appeared on my... ─────────────────────

→ Routed to: Billing

╭─────────────────────────────────────────── Billing Handler Response ────────────────────────────────────────────╮
│ **Hi [Your Name],**                                                                                             │
│                                                                                                                 │
│ I’m really sorry to hear that you were charged twice last month and that the duplicate charge isn’t showing up  │
│ in your dashboard. I understand how frustrating that can be, and I’m here to get this sorted for you quickly.   │
│                                                                                                                 │
│ ---                                                                                                             │
│                                                                                                                 │
│ ### What’s happening?                                                                                           │
│                                                                                                                 │
│ - **Billing cycle:** Our service runs on a **monthly billing cycle** that starts on the day you first           │
│ subscribed. All charges for a given cycle are posted to your card on the same day each month.                   │
│ - **Dashboard vs. card statement:** Occasionally a charge can appear on your card statement before it is        │
│ reflected in the dashboard (especially if the payment is still being processed). That’s why you might see a     │
│ duplicate on your card but only one entry in the portal.                                                        │
│                                                                                                                 │
│ ---                                                                                                             │
│                                                                                                                 │
│ ### How we’ll resolve this                                                                                      │
│                                                                                                                 │
│ 1. **Verify the duplicate:** I’ll pull the transaction details for the last billing cycle (the period from  to  │
│ ) and compare them with the charge you see on your card.                                                        │
│ 2. **Refund or credit:** If the duplicate is confirmed, I will:                                                 │
│    - **Issue an immediate refund** to the same card, which typically posts within 3‑5 business days, **or**     │
│    - **Apply a credit** to your next billing cycle if you prefer that option.                                   │
│ 3. **Confirmation:** I’ll send you a confirmation email with the refund reference number (or credit note) once  │
│ the action is completed.                                                                                        │
│                                                                                                                 │
│ ---                                                                                                             │
│                                                                                                                 │
│ ### What you can do right now                                                                                   │
│                                                                                                                 │
│ - **Download your invoices:**                                                                                   │
│   1. Log in to your account.                                                                                    │
│   2. Go to **Settings → Billing → Invoices**.         

───────────────────── Ticket: The CSV export button does nothing when I click it in Chrome... ─────────────────────

→ Routed to: Technical Bug

╭──────────────────────────────────────── Technical Bug Handler Response ─────────────────────────────────────────╮
│ **Thank you for reaching out!** I’m sorry you’re having trouble with the CSV export button. Let’s gather a bit  │
│ more information so I can troubleshoot effectively and get this resolved for you.                               │
│                                                                                                                 │
│ ---                                                                                                             │
│                                                                                                                 │
│ ### 1. Environment Details                                                                                      │
│ 1. **Chrome version** (you can find this under *Chrome → About Google Chrome*).                                 │
│ 2. **macOS version** (e.g., macOS 14.0, 14.1, etc.).                                                            │
│                                                                                                                 │
│ ### 2. Steps to Reproduce                                                                                       │
│ Please let me know the exact sequence you follow, for example:                                                  │
│ 1. Log in → navigate to ****.                                                                                   │
│ 2. Apply any filters or selections.                                                                             │
│ 3. Click the **CSV Export** button.                                                                             │
│                                                                                                                 │
│ If there are any variations (e.g., different pages, after a specific action), please include those as well.     │
│                                                                                                                 │
│ ### 3. Observed Behavior & Error Messages                                                                       │
│ - Does anything happen at all (e.g., a spinner, a toast notification, a network request in the DevTools         │
│ console)?                                                                                                       │
│ - Do you see any error messages in the UI or in the browser console (press `Cmd+Option+I` → **Console** tab)?   │
│                                                                                                                 │
│ ---                                                                                                             │
│                                                                                                                 │
│ ## Immediate Workarounds (while we investigate)                                                                 │
│                                                                                                                 │
│ 1. **Try a Private/Incognito window** – this disables extensions that might interfere.                          │
│ 2. **Disable any CSV‑related browser extensions** (e.g., download managers, ad blockers) and retry.             │
│ 3. **Use a different browser** (Safari or Firefox) to confirm whether the issue is Chrome‑specific.             │
│ 4. **Clear Chrome cache for this site**: Settings → Privacy & Security → Clear browsing data → “Cached images   │
│ and files” (select “All time” for the site).                                                                    │
│ 5. **Force a download via the API** (if you have developer tools access):                                       │
│    - Open DevTools (`Cmd+Option+I`), go to the **Network** tab, click the CSV button, and look for a request    │
│ ending in `.csv`. Right‑click the request → “Copy → Co

───────────────────── Ticket: Would be amazing if you added dark mode! It would make night... ─────────────────────

→ Routed to: Feature Request

╭─────────────────────────────────────── Feature Request Handler Response ────────────────────────────────────────╮
│ Thank you for taking the time to share this idea—dark mode is a popular request and we can see how it would     │
│ improve the experience for night‑time users.                                                                    │
│                                                                                                                 │
│ When we evaluate new features, we look at a combination of factors: the size and frequency of the user need,    │
│ the impact on overall product usability, technical effort, and how the request aligns with our current roadmap  │
│ and strategic goals. All incoming suggestions are logged in our feedback database, where they are reviewed      │
│ regularly by the product and engineering teams.                                                                 │
│                                                                                                                 │
│ To help us assess the value of dark mode for our community, could you tell us a bit more about how you’d use    │
│ it? For example:                                                                                                │
│ - Which parts of the application you’d most like to see in dark mode (e.g., dashboards, editor, settings        │
│ pages).                                                                                                         │
│ - Any specific accessibility or visual‑comfort challenges you encounter during night‑time use.                  │
│ - How many of your teammates or users would benefit from it.                                                    │
│                                                                                                                 │
│ Your details will be added to the feedback record and considered during our next prioritization cycle. Thanks   │
│ again for the suggestion!                                                                                       │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [6]:
# ── EXAMPLE 7d · Parallel decomposition — Code review pipeline ───────────────
# Run multiple specialized reviewers in parallel (independent → merge)
# In production this would use asyncio; here we show the pattern synchronously.

code_to_review = """\
import requests

def get_user_data(user_id):
    url = f"http://api.internal.com/users/{user_id}"
    response = requests.get(url, timeout=30)
    data = response.json()
    
    # Cache in memory for performance
    cache = {}
    cache[user_id] = data
    
    password = data.get('password_hash')
    print(f"Retrieved user {user_id}: password={password}")
    
    return data

def process_users(user_ids):
    results = []
    for uid in user_ids:
        result = get_user_data(uid)
        results.append(result)
    return results
"""

# Three specialized reviewers — run independently, then merge
reviewers = {
    "Security": "You are a cybersecurity expert. Review ONLY for security vulnerabilities, data leaks, and unsafe patterns. Ignore style.",
    "Performance": "You are a performance engineer. Review ONLY for inefficiencies, unnecessary operations, and scalability issues. Ignore security.",
    "Code Quality": "You are a senior engineer focused on code quality. Review ONLY for maintainability, error handling, and best practices. Ignore performance.",
}

review_results = {}
for role, system in reviewers.items():
    review_prompt = f"Review this Python code:\n```python\n{code_to_review}\n```\nProvide 3 specific findings with severity (Critical/High/Medium/Low) and a one-line fix recommendation each."
    review_results[role] = chat([
        {"role": "system", "content": system},
        {"role": "user", "content": review_prompt}
    ], temperature=0.1)

# Merge: synthesize all reviews into a prioritized action list
all_reviews = "\n\n".join(f"=== {role} Review ===\n{review}" for role, review in review_results.items())
merge_prompt = f"""\
You received three independent code reviews (security, performance, quality).
Merge them into a single prioritized action list.

Rules:
- Eliminate duplicates
- Sort by priority: Critical > High > Medium > Low
- Format each item: [PRIORITY] Description — Fix in one line
- Add a one-line "Ship it?" verdict at the top

Reviews:
{all_reviews}
"""

merged = chat([{"role": "user", "content": merge_prompt}], temperature=0.1)

for role, review in review_results.items():
    show(review, f"{role} Reviewer", "yellow")
print()
show(merged, "Merged & Prioritized Code Review (Parallel Chain)", "green")

╭─────────────────────────────────────────────── Security Reviewer ───────────────────────────────────────────────╮
│ **Finding 1 – Insecure transport (HTTP)**                                                                       │
│ - **Severity:** High                                                                                            │
│ - **Issue:** The code contacts the internal API over plain‑text HTTP (`http://api.internal.com/...`). An        │
│ attacker on the same network could intercept or tamper with the request/response (including the password hash). │
│ - **Fix:** Switch to HTTPS and verify the server’s certificate (e.g., `requests.get(url, timeout=30,            │
│ verify=True)`), and enforce TLS‑only communication for all internal services.                                   │
│                                                                                                                 │
│ **Finding 2 – Sensitive data leakage via logging**                                                              │
│ - **Severity:** Critical                                                                                        │
│ - **Issue:** The password hash retrieved from the API is printed to stdout (`print(f"Retrieved user {user_id}:  │
│ password={password}")`). This can end up in logs, console history, or monitoring tools, exposing credential     │
│ material.                                                                                                       │
│ - **Fix:** Remove the print statement or mask/redact the password hash (e.g., `print(f"Retrieved user           │
│ {user_id}: password=***")`) and ensure no sensitive fields are logged.                                          │
│                                                                                                                 │
│ **Finding 3 – Lack of request error handling and validation**                                                   │
│ - **Severity:** Medium                                                                                          │
│ - **Issue:** The code assumes the request succeeds and that `response.json()` returns a valid dict. Network     │
│ failures, non‑200 status codes, or malformed JSON will raise uncaught exceptions, potentially causing           │
│ denial‑of‑service or exposing stack traces.                                                                     │
│ - **Fix:** Add proper error handling and response validation (e.g., check `response.status_code == 200` and     │
│ wrap `response.json()` in a try/except block, returning a safe error object or raising a controlled exception). │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── Performance Reviewer ──────────────────────────────────────────────╮
│ **Finding 1 – Ineffective in‑memory cache (Medium)**                                                            │
│ The `cache` dictionary is created inside `get_user_data`, so it is discarded after each call and never actually │
│ caches anything.                                                                                                │
│                                                                                                                 │
│ *Fix:* Define a module‑level cache (or use `functools.lru_cache`) that persists across calls and is thread‑safe │
│ if needed.                                                                                                      │
│                                                                                                                 │
│ ---                                                                                                             │
│                                                                                                                 │
│ **Finding 2 – Serial HTTP requests limit scalability (High)**                                                   │
│ `process_users` fetches each user one after another; the total latency grows linearly with the number of IDs    │
│ and blocks the thread.                                                                                          │
│                                                                                                                 │
│ *Fix:* Parallelise the calls (e.g., with `concurrent.futures.ThreadPoolExecutor` or `asyncio` + `httpx`) or     │
│ batch the IDs if the API supports bulk retrieval.                                                               │
│                                                                                                                 │
│ ---                                                                                                             │
│                                                                                                                 │
│ **Finding 3 – Re‑creating a new TCP connection for every request (Low‑Medium)**                                 │
│ `requests.get` builds a fresh `Session` each time, incurring DNS lookup, TLS handshake, and TCP‑handshake       │
│ overhead for every user.                                                                                        │
│                                                                                                                 │
│ *Fix:* Create a single `requests.Session` (or reuse a session pool) and pass it to `get_user_data` so           │
│ connections are kept alive across calls.                                                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── Code Quality Reviewer ─────────────────────────────────────────────╮
│ **Finding 1 – Sensitive data exposure (Critical)**                                                              │
│ *Issue*: The function prints the user’s password hash (`print(f"Retrieved user {user_id}:                       │
│ password={password}")`).                                                                                        │
│ *Fix*: Remove the print statement or mask/redact the password before logging (e.g., `logger.debug("Retrieved    │
│ user %s", user_id)`).                                                                                           │
│                                                                                                                 │
│ **Finding 2 – Missing error handling for HTTP/JSON failures (High)**                                            │
│ *Issue*: `requests.get` and `response.json()` are called without checking for network errors, non‑200 status    │
│ codes, or malformed JSON, which can raise uncaught exceptions.                                                  │
│ *Fix*: Wrap the request in a `try/except` block, verify `response.raise_for_status()`, and handle               │
│ `requests.RequestException` and `json.JSONDecodeError` gracefully.                                              │
│                                                                                                                 │
│ **Finding 3 – Ineffective in‑function caching (Low)**                                                           │
│ *Issue*: `cache = {}` is created inside `get_user_data` on every call, so the data is never reused and the      │
│ cache is lost after the function returns.                                                                       │
│ *Fix*: Move the cache to module scope (or use `functools.lru_cache`) so it persists across calls, e.g., `cache  │
│ = {}` defined at the top of the module and referenced inside the function.                                      │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────── Merged & Prioritized Code Review (Parallel Chain) ───────────────────────────────╮
│ Ship it? Not yet – resolve the critical and high‑severity findings before release.                              │
│                                                                                                                 │
│ [Critical] Password hash printed to stdout/logs exposing credentials — Remove the print or mask the hash and    │
│ use a proper logger without sensitive data.                                                                     │
│ [High] Serial user fetches block the thread and cause linear latency — Parallelise HTTP calls with              │
│ ThreadPoolExecutor/asyncio or batch requests.                                                                   │
│ [High] Missing/insufficient HTTP and JSON error handling leads to crashes — Wrap requests in try/except, call   │
│ `response.raise_for_status()`, and handle `JSONDecodeError`.                                                    │
│ [High] Insecure HTTP transport to internal API — Switch to HTTPS with certificate verification and enforce      │
│ TLS‑only communication.                                                                                         │
│ [Medium] Cache defined inside the function is discarded each call, preventing reuse — Move the cache to module  │
│ scope or apply `functools.lru_cache`, ensuring thread‑safety.                                                   │
│ [Medium] New `requests.Session` created per call causing repeated TCP/TLS handshakes — Instantiate a single     │
│ Session (or connection pool) and reuse it across calls.                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

### 🧠 Student Exercise 7
Build a **3-step blog post pipeline**:
- **Step 1**: Given a topic (e.g., "The impact of AI on healthcare"), generate 5 different headline options
- **Step 2**: Score each headline on: clickability (1-10), clarity (1-10), SEO potential (1-10). Pick the best one.
- **Step 3**: Using the winning headline, generate a 400-word blog post outline (title + 5 sections with bullet points)

Print the output of each step with a clear separator between them.

---
## Part 8 — Meta-Prompting and Self-Refinement Loops

### What is meta-prompting?
**Meta-prompting** = using an LLM to **write, evaluate, or improve prompts** — rather than writing them by hand.

### Self-refinement loop pattern
```
1. GENERATE   → LLM produces an initial response
2. CRITIQUE   → LLM (or same LLM) evaluates the response against criteria
3. REFINE     → LLM revises the response based on the critique
4. (repeat N times or until quality threshold met)
```

### Key variants
| Variant | Description | Best for |
|---------|------------|----------|
| **Generate-Critique-Revise** | Single model, 3-step loop | Writing, code quality |
| **Self-Consistency** | Generate N responses, pick majority | Factual Q&A, math |
| **Meta-Prompt Generator** | Generate → evaluate prompts themselves | Prompt optimization |
| **Constitutional Refinement** | Critique against a set of principles | Safety-critical outputs |

### Research insight
> Self-refinement consistently improves quality by **10–25%** over single-pass generation. *(Lushbinary, 2026)*  
> Meta-prompting that writes prompts can match expert-written prompts on 70% of tasks. *(SurePrompts, 2026)*

In [8]:
# ── EXAMPLE 8a · Meta-Prompt Generator — LLM writes better prompts ───────────
# Give the model a weak task description → ask it to generate the optimal prompt

weak_description = "I want a prompt that makes the AI help me write marketing emails."

meta_prompt = f"""\
You are a prompt engineering expert. Your job is to take a vague task description 
and generate a high-quality, production-ready prompt for it.

A high-quality prompt includes:
1. Clear system role / persona
2. Specific task instruction (not vague)
3. Required input placeholders (use {{{{variable_name}}}} syntax)
4. Explicit output format with structure
5. 2-3 example outputs showing the expected style
6. Quality constraints (tone, length, what to avoid)

Task description from user:
"{weak_description}"

Generate the complete, production-ready prompt. Put it inside <prompt></prompt> tags.
After the tags, add a 3-bullet explanation of your design decisions.
"""

result = chat([{"role": "user", "content": meta_prompt}], temperature=0.4)
show(result, "Meta-Prompt Generator Output", "magenta")

# Extract and test the generated prompt
import re
generated_prompt = re.search(r'<prompt>(.*?)</prompt>', result, re.DOTALL)
if generated_prompt:
    print("\n✓ Prompt extracted. Testing it...")
    test_input = generated_prompt.group(1).strip()
    # Replace placeholders with actual values for a quick test
    test_prompt = test_input.replace("{product}", "CloudSync Pro").replace("{audience}", "small business owners")\
                            .replace("{goal}", "free trial signup").replace("{tone}", "professional but friendly")
    test_output = chat([{"role": "user", "content": test_prompt}], temperature=0.5)
    show(test_output[:800] + "..." if len(test_output) > 800 else test_output,
         "Testing the Generated Prompt", "green")

╭───────────────────────────────────────── Meta-Prompt Generator Output ──────────────────────────────────────────╮
│ <prompt>                                                                                                        │
│ **System Role / Persona**                                                                                       │
│ You are a seasoned B2B/B2C copywriter specializing in high‑conversion marketing emails. You understand audience │
│ segmentation, value proposition framing, and compliance with email best‑practice guidelines.                    │
│                                                                                                                 │
│ **Task Instruction**                                                                                            │
│ Using the information provided in the input placeholders, craft a complete marketing email ready for            │
│ deployment. The email must include a subject line, a preheader, a body (with an opening hook, key benefit       │
│ bullets, and a clear call‑to‑action), and a professional sign‑off.                                              │
│                                                                                                                 │
│ **Required Input Placeholders**                                                                                 │
│ - {{product_name}} – Name of the product or service.                                                            │
│ - {{target_audience}} – Brief description of the ideal recipient (e.g., “small‑business owners in the retail    │
│ sector”).                                                                                                       │
│ - {{key_benefits}} – 3‑5 bullet‑point benefits of the product, each no longer than 12 words.                    │
│ - {{unique_offer}} – Any special promotion, discount, or limited‑time offer.                                    │
│ - {{call_to_action}} – Desired action (e.g., “Schedule a demo”, “Claim your free trial”).                       │
│ - {{sender_name}} – Name of the sender.                                                                         │
│ - {{sender_title}} – Sender’s title/role.                                                                       │
│ - {{company_name}} – Your company’s name.                                                                       │
│ - {{company_website}} – URL to the company’s website.                                                           │
│                                                                                                                 │
│ **Output Format**                                                                                               │
│ Return the email in **Markdown** with the following structure:                                                  │
│                                                                                                                 │
│ ```                                                                                                             │
│ **Subject:** {{subject_line}}                                                                                   │
│ **Preheader:** {{preheader_text}}                                                                               │
│                                                                                                                 │
│ Hi {{target_audience}},                                                                                         │
│                                                                                                                 │
│ {{opening_paragraph}}                                                                                           │
│                                                                                                                 │
│ **Why {{product_name}}?**                             


✓ Prompt extracted. Testing it...


╭───────────────────────────────────────── Testing the Generated Prompt ──────────────────────────────────────────╮
│ **Subject:** {{subject_line}}                                                                                   │
│ **Preheader:** {{preheader_text}}                                                                               │
│                                                                                                                 │
│ Hi {{target_audience}},                                                                                         │
│                                                                                                                 │
│ {{opening_paragraph}}                                                                                           │
│                                                                                                                 │
│ **Why {{product_name}}?**                                                                                       │
│ - {{key_benefit_1}}                                                                                             │
│ - {{key_benefit_2}}                                                                                             │
│ - {{key_benefit_3}}                                                                                             │
│ {{#if key_benefit_4}}- {{key_benefit_4}}{{/if}}                                                                 │
│ {{#if key_benefit_5}}- {{key_benefit_5}}{{/if}}                                                                 │
│                                                                                                                 │
│ {{unique_offer_paragraph}}                                                                                      │
│                                                                                                                 │
│ Ready to {{call_to_action}}? Click here: {{cta_url}}                                                            │
│                                                                                                                 │
│ Best regards,                                                                                                   │
│ {{sender_name}}                                                                                                 │
│ {{sender_title}} | {{company_name}}                                                                             │
│ {{company_website}}                                                                                             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [9]:
# ── EXAMPLE 8b · Generate-Critique-Revise Loop — Essay quality improvement ───
# This is the most common self-refinement pattern.
# Each iteration the essay gets better. We'll run 2 refinement rounds.

essay_topic = "The role of curiosity in scientific discovery"

# Critique criteria — the model will evaluate against these
critique_criteria = """\
Evaluate the essay on these 5 criteria (score 1–10 each, then give specific improvement notes):
1. Thesis clarity — Is the main argument crystal clear in the opening paragraph?
2. Evidence quality — Are claims supported with specific examples, not vague generalizations?
3. Flow — Do paragraphs connect logically? Are transitions smooth?
4. Originality — Does it say anything surprising or non-obvious?
5. Conclusion impact — Does it leave the reader with something to think about?

For each criterion scoring below 7, provide ONE specific, actionable revision note.
Format: [Criterion]: Score/10 — Note (if score < 7)
"""

# Round 0: Generate initial draft
rprint(Rule("[bold]Round 0 — Initial Draft[/]"))
generate_prompt = f"Write a 300-word essay on: '{essay_topic}'. Focus on depth over breadth."
essay = chat([{"role": "user", "content": generate_prompt}], temperature=0.7)
show(essay, "Draft v0 (Initial)", "yellow")

# Refinement loop
for round_num in range(1, 3):  # 2 refinement rounds
    rprint(Rule(f"[bold]Round {round_num} — Critique & Revise[/]"))
    
    # Critique
    critique = chat([{
        "role": "user",
        "content": f"Read this essay and evaluate it:\n\n{essay}\n\n{critique_criteria}"
    }], temperature=0.1)
    show(critique, f"Critique (Round {round_num})", "red")
    
    # Revise
    essay = chat([{
        "role": "user",
        "content": f"""\
Here is an essay and specific critique feedback.
Rewrite the essay to address EVERY critique point below.
Keep the same length (~300 words). Do not just add filler.

Original essay:
{essay}

Critique to address:
{critique}

Revised essay:"""
    }], temperature=0.5)
    show(essay, f"Revised Essay v{round_num}", "green")

───────────────────────────────────────────── Round 0 — Initial Draft ─────────────────────────────────────────────

╭────────────────────────────────────────────── Draft v0 (Initial) ───────────────────────────────────────────────╮
│ Curiosity is the engine that propels the scientific mind from wonder to knowledge, converting a fleeting sense  │
│ of incompleteness into systematic inquiry. Unlike mere interest, curiosity is an affective‑cognitive tension:   │
│ it registers a gap between what is known and what is imagined, compelling the brain to reduce the disparity.    │
│ This “information‑seeking” drive shapes every stage of discovery, from the formulation of a question to the     │
│ interpretation of data.                                                                                         │
│                                                                                                                 │
│ At the conceptual level, curiosity generates hypotheses. When a researcher encounters an anomalous              │
│ observation—a particle that deviates from expected trajectories, a species that thrives in an extreme niche—the │
│ emotional discomfort of the unknown triggers a mental search for explanatory patterns. This search is not       │
│ random; it leverages existing schemas, recombining them in novel ways. The resulting conjecture is a concrete   │
│ target for empirical testing, turning vague wonder into a testable proposition.                                 │
│                                                                                                                 │
│ During experimentation, curiosity sustains perseverance. Scientific work is riddled with false leads,           │
│ methodological setbacks, and ambiguous results. The same affective tension that sparked the initial question    │
│ fuels a willingness to iterate, to redesign apparatus, or to explore alternative models. Moreover, curiosity    │
│ shapes the choice of methods: a curious mind is more inclined to employ unconventional techniques, to probe     │
│ peripheral variables, or to adopt interdisciplinary tools, thereby expanding the methodological repertoire of a │
│ field.                                                                                                          │
│                                                                                                                 │
│ Interpretation illustrates curiosity’s deeper role. When data resist straightforward explanation, a curious     │
│ scientist resists the comfort of premature closure. Instead, they engage in “reflective curiosity,”             │
│ scrutinizing assumptions, considering hidden biases, and entertaining counter‑intuitive explanations. This      │
│ meta‑cognitive stance often leads to paradigm‑shifting insights, as exemplified by the realization that quantum │
│ phenomena defy classical determinism.                                                                           │
│                                                                                                                 │
│ Historically, breakthroughs—from Mendel’s pea experiments to the discovery of CRISPR’s adaptive immunity—share  │
│ a common narrative: a persistent, open‑ended curiosity that refused to accept the status quo. In sum, curiosity │
│ is not a peripheral motivator but the structural backbone of scientific discovery, converting the discomfort of │
│ the unknown into a disciplined, creative pursuit of truth.                                                      │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

─────────────────────────────────────────── Round 1 — Critique & Revise ───────────────────────────────────────────

╭────────────────────────────────────────────── Critique (Round 1) ───────────────────────────────────────────────╮
│ **Thesis clarity**: 8/10                                                                                        │
│                                                                                                                 │
│ **Evidence quality**: 6/10 — Add concrete, cited examples (e.g., specific study dates, researchers, or          │
│ experimental results) rather than generic references to “a particle” or “quantum phenomena” to substantiate     │
│ each claim.                                                                                                     │
│                                                                                                                 │
│ **Flow**: 7/10                                                                                                  │
│                                                                                                                 │
│ **Originality**: 6/10 — Introduce a less‑expected angle—such as how curiosity drives breakthroughs in fields    │
│ like artificial intelligence or climate modeling—to give the essay a more distinctive voice.                    │
│                                                                                                                 │
│ **Conclusion impact**: 6/10 — Close with a provocative question, a striking quotation, or a call‑to‑action that │
│ challenges readers to nurture curiosity in their own scientific practice.                                       │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── Revised Essay v1 ────────────────────────────────────────────────╮
│ Curiosity is the engine that drives scientific discovery, turning the discomfort of an unanswered question into │
│ a disciplined quest for truth. It is not merely a pleasant interest; it is an affective‑cognitive tension that  │
│ registers a gap between what is known and what can be imagined, compelling the mind to close that gap. This     │
│ tension underlies every stage of research, from hypothesis formation to the reinterpretation of data.           │
│                                                                                                                 │
│ At the conceptual level, curiosity births hypotheses. When James Clerk Maxwell observed that light behaved like │
│ an electromagnetic wave in 1865, the puzzling “action‑at‑a‑distance” of electricity sparked a mental search     │
│ that produced his equations of electromagnetism (Maxwell, 1865). Similarly, the unexpected excess of events at  │
│ 125 GeV in the Large Hadron Collider led the ATLAS and CMS collaborations to confirm the Higgs boson in 2012, a │
│ breakthrough driven by the uneasy feeling that the Standard Model was incomplete (Aad et al., 2012; Chatrchyan  │
│ et al., 2012).                                                                                                  │
│                                                                                                                 │
│ During experimentation, curiosity fuels perseverance. The 2012 CRISPR breakthrough began with Jennifer Doudna’s │
│ curiosity about a bacterial immune system; after years of failed assays, her team finally demonstrated          │
│ RNA‑guided DNA cleavage (Jinek et al., 2012). In artificial intelligence, DeepMind’s curiosity‑driven           │
│ reinforcement learning algorithm AlphaGo chose unconventional moves that baffled human experts, ultimately      │
│ defeating world champion Lee Sedol in 2016 (Silver et al., 2016). These cases show how curiosity prompts        │
│ researchers to redesign apparatus, adopt interdisciplinary tools, and persist through false leads.              │
│                                                                                                                 │
│ Interpretation reveals curiosity’s deeper role. When Bell’s theorem experiments in 1964 produced correlations   │
│ that defied classical intuition, John Bell’s “reflective curiosity” forced physicists to confront hidden        │
│ assumptions, paving the way for quantum information science (Bell, 1964). In climate science, the 2021 CMIP6    │
│ intercomparison revealed unexpected feedbacks in cloud modeling, prompting a wave of curiosity‑driven revisions │
│ that improve future projections (Eyring et al., 2021).                                                          │
│                                                                                                                 │
│ Across fields—from Mendel’s peas (1866) to modern AI—curiosity is the structural backbone of progress. As       │
│ Albert Einstein warned, “The important thing is not to stop questioning.” Will you let that question guide your │
│ next experiment?                                                                                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

─────────────────────────────────────────── Round 2 — Critique & Revise ───────────────────────────────────────────

╭────────────────────────────────────────────── Critique (Round 2) ───────────────────────────────────────────────╮
│ **Thesis clarity**: 8/10 —                                                                                      │
│                                                                                                                 │
│ **Evidence quality**: 6/10 — *Add brief contextual explanation for each example (e.g., why Maxwell’s            │
│ “action‑at‑a‑distance” problem directly led to his equations) and verify dates/attributions to avoid factual    │
│ slips that weaken credibility.*                                                                                 │
│                                                                                                                 │
│ **Flow**: 7/10 —                                                                                                │
│                                                                                                                 │
│ **Originality**: 5/10 — *Introduce a less‑expected perspective—such as how institutional incentives or          │
│ failure‑aversion can suppress curiosity—and discuss how overcoming those barriers uniquely fuels                │
│ breakthroughs.*                                                                                                 │
│                                                                                                                 │
│ **Conclusion impact**: 6/10 — *End with a concrete, forward‑looking prompt (e.g., a specific open question in   │
│ your field) rather than a generic quote, so readers leave with a tangible “next experiment” to contemplate.*    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── Revised Essay v2 ────────────────────────────────────────────────╮
│ **Thesis:**  Curiosity is the indispensable engine of scientific breakthroughs, and its impact is magnified     │
│ when researchers push past institutional incentives and a culture that penalizes failure.                       │
│                                                                                                                 │
│ At the conceptual stage, curiosity turns a perceived gap into a testable hypothesis.  Maxwell’s unease with the │
│ “action‑at‑a‑distance” description of electricity—an idea that forces forces to act instantaneously across      │
│ empty space—led him to ask whether electric and magnetic fields could be unified.  By reformulating the problem │
│ in terms of continuous fields, his 1865 papers produced the set of equations that later revealed light as an    │
│ electromagnetic wave (Maxwell, 1865).  Likewise, the excess of events near 125 GeV observed by the ATLAS and    │
│ CMS detectors in 2012 signaled that the Standard Model might be incomplete; the curiosity‑driven analysis of    │
│ this anomaly culminated in the discovery of the Higgs boson (Aad et al., 2012; Chatrchyan et al., 2012).        │
│                                                                                                                 │
│ During experimentation, curiosity sustains perseverance against setbacks.  Doudna’s intrigue with a bacterial   │
│ immune system—CRISPR‑Cas9—prompted a series of failed cleavage assays before the 2012 Science report            │
│ demonstrated programmable RNA‑guided DNA cutting (Jinek et al., 2012).  In artificial intelligence, DeepMind’s  │
│ reinforcement‑learning agent AlphaGo was programmed to explore “novel” moves rather than merely imitate human   │
│ play; this curiosity‑driven exploration produced the unexpected “hand‑of‑god” move that defeated Lee Sedol in   │
│ 2016 (Silver et al., 2016).                                                                                     │
│                                                                                                                 │
│ Interpretation reveals curiosity’s deeper role in reshaping theory.  Bell’s 1964 theorem exposed a tension      │
│ between quantum predictions and classical locality; subsequent experiments in the 1970s that confirmed          │
│ Bell‑type correlations forced physicists to abandon hidden‑variable assumptions and seeded quantum information  │
│ science (Bell, 1964).  In climate modeling, the 2021 CMIP6 intercomparison uncovered anomalous cloud‑feedback   │
│ behavior, sparking a wave of curiosity‑driven model revisions that now improve projections of future warming    │
│ (Eyring et al., 2021).                                                                                          │
│                                                                                                                 │
│ Yet curiosity often collides with institutional pressures that reward safe, incremental work and punish         │
│ failure.  When researchers deliberately allocate protected time for high‑risk questions—such as the NIH’s       │
│ “high‑risk, high‑reward” grants or university seed‑fund programs—they create the space needed for the very      │
│ uncertainty that fuels discovery.                                                                               │
│                                                                                                                 │
│ **Forward‑looking prompt:**  In synthetic biology, can we design a self‑optimizing gene circuit that learns     │
│ from failed expression attempts and autonomously rewrites its regulatory logic?  Formulating and testing that   │
│ question will let curiosity, unshackled from risk‑averse norms, drive the next leap forward.                    │
╰───────────────────────────────────────────────────────

In [10]:
# ── EXAMPLE 8c · Self-Consistency Voting — for factual reliability ────────────
# Generate N independent answers, then pick the majority answer.
# Especially powerful for questions with one correct answer (math, factual recall).
# Research: +17.9% accuracy on GSM8K (Wang et al., 2022)

from collections import Counter

def self_consistency(question: str, n_samples: int = 5, temperature: float = 0.7) -> dict:
    """
    Generate n_samples independent answers and return the majority answer
    along with confidence (fraction of votes for winner).
    """
    answers = []
    for i in range(n_samples):
        prompt = f"""\
Think through this step by step, then give ONLY the final numerical answer 
(just the number, no units or explanation).

Question: {question}
Final answer:"""
        answer = chat([{"role": "user", "content": prompt}], temperature=temperature)
        # Extract just the number
        number = re.search(r'-?\d+\.?\d*', answer.strip())
        if number:
            answers.append(number.group())
        print(f"  Sample {i+1}: {answer.strip()[:30]}")
    
    vote_counts = Counter(answers)
    winner, count = vote_counts.most_common(1)[0]
    confidence = count / len(answers)
    return {"answer": winner, "confidence": confidence, "all_votes": dict(vote_counts)}


# Test on a problem where direct prompting often fails
problem = """\
A store offers 20% off, then an additional 15% off the discounted price. 
A customer also has a $10 coupon applied after both discounts.
Original price: $200. What is the final price the customer pays?
"""

print("Running self-consistency (5 independent samples)...")
result = self_consistency(problem, n_samples=5)

print(f"\n📊 Vote distribution: {result['all_votes']}")
print(f"✓ Majority answer: {result['answer']}")
print(f"📈 Confidence: {result['confidence']*100:.0f}%")

# The correct answer: $200 × 0.80 × 0.85 - $10 = $126
print(f"\n✅ Correct answer: $126 (200 × 0.80 × 0.85 − 10)")

Running self-consistency (5 independent samples)...
  Sample 1: 126
  Sample 2: 126
  Sample 3: 126
  Sample 4: 126
  Sample 5: 126

📊 Vote distribution: {'126': 5}
✓ Majority answer: 126
📈 Confidence: 100%

✅ Correct answer: $126 (200 × 0.80 × 0.85 − 10)


In [11]:
# ── EXAMPLE 8d · Prompt Optimization Loop — meta-prompting to improve prompts ─
# This is a full meta-prompting loop:
# 1. Start with a prompt
# 2. Test it on examples
# 3. Have the model critique the prompt (not the output)
# 4. Generate an improved prompt
# 5. Repeat

# The task: classify email urgency as High/Medium/Low
test_cases = [
    {"email": "Server is down, 500 errors, users can't log in!", "expected": "High"},
    {"email": "Can we schedule a call next week to discuss Q4 planning?", "expected": "Low"},
    {"email": "The CSV export is slightly misaligned in Firefox. Not urgent.", "expected": "Low"},
    {"email": "Payment processing is failing for enterprise clients. Revenue impact.", "expected": "High"},
    {"email": "Could you update the team page with my new photo when you get a chance?", "expected": "Low"},
]

def evaluate_prompt(prompt_template: str, cases: list[dict]) -> tuple[float, list[dict]]:
    """Score a prompt template on test cases. Returns (accuracy, results)."""
    results = []
    for case in cases:
        full_prompt = prompt_template + f"\n\nEmail: \"{case['email']}\"\nUrgency:"
        output = chat([{"role": "user", "content": full_prompt}], temperature=0.0).strip()
        correct = case["expected"].lower() in output.lower()
        results.append({"email": case["email"][:40], "expected": case["expected"],
                        "got": output[:20], "correct": correct})
    accuracy = sum(r["correct"] for r in results) / len(results)
    return accuracy, results


# Round 0: Weak initial prompt
current_prompt = "Classify the urgency of this email. Reply: High, Medium, or Low."
acc, results = evaluate_prompt(current_prompt, test_cases)
print(f"Round 0 prompt accuracy: {acc*100:.0f}%")
rprint(results)

# Round 1: Ask the model to improve the prompt
failures = [r for r in results if not r["correct"]]
optimize_prompt = f"""\
You are a prompt engineer. I have a prompt for classifying email urgency:

CURRENT PROMPT:
\"\"\"
{current_prompt}
\"\"\"

TEST RESULTS:
Accuracy: {acc*100:.0f}%
Failures: {failures}

TASK: Rewrite the prompt to fix these failures. 
Add clear criteria for High/Medium/Low (1-2 sentences each).
Keep it under 120 words. Output only the improved prompt, nothing else.
"""

improved_prompt = chat([{"role": "user", "content": optimize_prompt}], temperature=0.3)
show(improved_prompt, "Round 1 — Meta-Generated Improved Prompt", "blue")

# Test improved prompt
acc2, results2 = evaluate_prompt(improved_prompt, test_cases)
print(f"\nRound 1 prompt accuracy: {acc2*100:.0f}% (was {acc*100:.0f}%)")
rprint(results2)

Round 0 prompt accuracy: 80%


[
    {'email': "Server is down, 500 errors, users can't ", 'expected': 'High', 'got': 'High', 'correct': True},
    {'email': 'Can we schedule a call next week to disc', 'expected': 'Low', 'got': 'Medium', 'correct': False},
    {'email': 'The CSV export is slightly misaligned in', 'expected': 'Low', 'got': 'Low', 'correct': True},
    {'email': 'Payment processing is failing for enterp', 'expected': 'High', 'got': 'High', 'correct': True},
    {'email': 'Could you update the team page with my n', 'expected': 'Low', 'got': 'Low', 'correct': True}
]

╭─────────────────────────────────── Round 1 — Meta-Generated Improved Prompt ────────────────────────────────────╮
│ Classify the urgency of the email as High, Medium, or Low.                                                      │
│ High: requires immediate action within 24 hours, involves critical deadlines, emergencies, or potential severe  │
│ consequences.                                                                                                   │
│ Medium: important but can be addressed within 2–3 days, includes requests, meetings, or non‑critical issues.    │
│ Low: informational or routine, can be handled after several days without impact.                                │
│ Respond with only the word High, Medium, or Low.                                                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


Round 1 prompt accuracy: 80% (was 80%)


[
    {'email': "Server is down, 500 errors, users can't ", 'expected': 'High', 'got': 'High', 'correct': True},
    {'email': 'Can we schedule a call next week to disc', 'expected': 'Low', 'got': 'Medium', 'correct': False},
    {'email': 'The CSV export is slightly misaligned in', 'expected': 'Low', 'got': 'Low', 'correct': True},
    {'email': 'Payment processing is failing for enterp', 'expected': 'High', 'got': 'High', 'correct': True},
    {'email': 'Could you update the team page with my n', 'expected': 'Low', 'got': 'Low', 'correct': True}
]

In [12]:
# ── EXAMPLE 8e · Code Self-Review Loop — generate, test, fix ─────────────────
# The model generates code, then reviews it as a critic,
# then produces a final corrected version.

coding_task = """\
Write a Python function `find_duplicates(lst)` that:
- Takes a list of any comparable items
- Returns a list of items that appear MORE than once (no duplicates in the output list)
- Preserves the order of first occurrence in the original list
- Handles empty lists gracefully
"""

# Step 1: Generate initial code
rprint(Rule("[bold]Step 1 — Generate Initial Code[/]"))
initial_code = chat([
    {"role": "user", "content": f"Write the function described below. Only output code, no explanation.\n\n{coding_task}"}
], temperature=0.5)
show(initial_code, "Initial Code (Generated)", "yellow")

# Step 2: Self-critique the code
rprint(Rule("[bold]Step 2 — Self-Critique[/]"))
critique_prompt = f"""\
Review this Python function for correctness, edge cases, and code quality.

Task it should solve:
{coding_task}

Code to review:
{initial_code}

Critique using these lenses:
1. CORRECTNESS — Does it produce the right output? Trace through 2-3 examples mentally.
2. EDGE CASES — What about: empty list, all duplicates, single element, mixed types?
3. TIME COMPLEXITY — What is O(n)? Is there a more efficient approach?
4. CODE QUALITY — Type hints? Docstring? Readable variable names?

Be specific about any bugs found.
"""
critique = chat([{"role": "user", "content": critique_prompt}], temperature=0.1)
show(critique, "Self-Critique", "red")

# Step 3: Revise based on critique
rprint(Rule("[bold]Step 3 — Revised Final Code[/]"))
revise_prompt = f"""\
Based on this critique, write the final corrected version of the function.
Address every issue mentioned. Include a docstring and type hints.
Output only the final code.

Critique:
{critique}
"""
final_code = chat([{"role": "user", "content": revise_prompt}], temperature=0.2)
show(final_code, "Final Corrected Code (After Self-Refinement)", "green")

# Actually execute and test the final code
print("\n🧪 Executing final code to verify...")
try:
    # Extract code from markdown fences if present
    code_only = re.sub(r'```python\n|```', '', final_code).strip()
    exec(code_only, globals())
    
    test_cases_code = [
        ([], []),
        ([1, 2, 3], []),
        ([1, 2, 2, 3, 3, 3], [2, 3]),
        (["a", "b", "a", "c", "b"], ["a", "b"]),
        ([1], []),
    ]
    all_pass = True
    for inp, expected in test_cases_code:
        result = find_duplicates(inp)  # type: ignore
        passed = result == expected
        status = "✓" if passed else "✗"
        print(f"  {status} find_duplicates({inp}) = {result} (expected {expected})")
        if not passed:
            all_pass = False
    print(f"\n{'✅ All tests passed!' if all_pass else '❌ Some tests failed.'}")
except Exception as e:
    print(f"❌ Execution error: {e}")

───────────────────────────────────────── Step 1 — Generate Initial Code ──────────────────────────────────────────

╭─────────────────────────────────────────── Initial Code (Generated) ────────────────────────────────────────────╮
│ ```python                                                                                                       │
│ def find_duplicates(lst):                                                                                       │
│     """                                                                                                         │
│     Return a list of items that appear more than once in `lst`,                                                 │
│     preserving the order of their first occurrence.                                                             │
│     """                                                                                                         │
│     seen = {}                                                                                                   │
│     duplicates = []                                                                                             │
│     for item in lst:                                                                                            │
│         if item in seen:                                                                                        │
│             if not seen:          # first time we detect a duplicate                                            │
│                 duplicates.append(item)                                                                         │
│                 seen = True       # mark as already added to result                                             │
│         else:                                                                                                   │
│             seen = False          # seen once, not yet a duplicate                                              │
│     return duplicates                                                                                           │
│ ```                                                                                                             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

───────────────────────────────────────────── Step 2 — Self-Critique ──────────────────────────────────────────────

╭───────────────────────────────────────────────── Self-Critique ─────────────────────────────────────────────────╮
│ ## Quick Summary                                                                                                │
│                                                                                                                 │
│ | Aspect | Verdict | Comments |                                                                                 │
│ |--------|---------|----------|                                                                                 │
│ | **Correctness** | ✅ Mostly correct | Works for any **hashable** element, returns the right list in the right │
│ order. |                                                                                                        │
│ | **Edge‑case handling** | ✅ Good for the usual cases, but fails on **unhashable** items (e.g. lists, dicts) | │
│ Empty list, single‑element list, all‑duplicate list, mixed‑type hashable list are fine. |                       │
│ | **Time / Space** | ✅ O(n) time, O(n) extra space | No asymptotically better solution exists for a            │
│ single‑pass “first‑occurrence‑preserving” duplicate detection. |                                                │
│ | **Code quality** | ⚠️ Needs polishing | No type hints, docstring could be richer, variable names could be     │
│ clearer, and the implementation can be simplified. |                                                            │
│                                                                                                                 │
│ Below is a detailed walk‑through of each lens, followed by a revised implementation that addresses the          │
│ identified issues.                                                                                              │
│                                                                                                                 │
│ ---                                                                                                             │
│                                                                                                                 │
│ ## 1. Correctness                                                                                               │
│                                                                                                                 │
│ ### How the algorithm works                                                                                     │
│                                                                                                                 │
│ ```python                                                                                                       │
│ seen = {}          # key → bool (False = seen once, True = already added to result)                             │
│ duplicates = []    # output list                                                                                │
│ for item in lst:                                                                                                │
│     if item in seen:                                                                                            │
│         if not seen:          # first time we notice a second occurrence                                        │
│             duplicates.append(item)                                                                             │
│             seen = True       # remember that we have already emitted it                                        │
│     else:                                                                                                       │
│         seen = False          # first time we see the item                                                      │
│ ```                                                                                                             │
│                                                          

─────────────────────────────────────────── Step 3 — Revised Final Code ───────────────────────────────────────────

╭───────────────────────────────── Final Corrected Code (After Self-Refinement) ──────────────────────────────────╮
│ ```python                                                                                                       │
│ from __future__ import annotations                                                                              │
│                                                                                                                 │
│ from collections.abc import Iterable                                                                            │
│ from typing import List, Set, TypeVar                                                                           │
│                                                                                                                 │
│ T = TypeVar("T")                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
│ def find_duplicates(lst: Iterable[T]) -> List[T]:                                                               │
│     """                                                                                                         │
│     Return a list of items that appear more than once in ``lst``,                                               │
│     preserving the order of their first occurrence.                                                             │
│                                                                                                                 │
│     The function works only with **hashable** items (i.e. items that can be                                     │
│     used as keys in a ``dict`` or members of a ``set``).  If an unhashable                                      │
│     element is encountered, a ``TypeError`` is raised.                                                          │
│                                                                                                                 │
│     Parameters                                                                                                  │
│     ----------                                                                                                  │
│     lst : Iterable[T]                                                                                           │
│         The input collection.  It may be empty.                                                                 │
│                                                                                                                 │
│     Returns                                                                                                     │
│     -------                                                                                                     │
│     List[T]                                                                                                     │
│         A list containing each duplicated element exactly once,                                                 │
│         ordered by the position of its first appearance in ``lst``.                                             │
│                                                                                                                 │
│     Examples                                                                                                    │
│     --------                                                                                                    │
│     >>> find_duplicates([1, 2, 3, 2, 1, 4])                                                                     │
│     [2, 1]                                                                                                      │
│     >>> find_duplicates(['a', 'b', 'a', 'a'])         


🧪 Executing final code to verify...
All quick tests passed.
  ✓ find_duplicates([]) = [] (expected [])
  ✓ find_duplicates([1, 2, 3]) = [] (expected [])
  ✓ find_duplicates([1, 2, 2, 3, 3, 3]) = [2, 3] (expected [2, 3])
  ✓ find_duplicates(['a', 'b', 'a', 'c', 'b']) = ['a', 'b'] (expected ['a', 'b'])
  ✓ find_duplicates([1]) = [] (expected [])

✅ All tests passed!


### 🧠 Student Exercise 8
Build a **3-round self-refinement loop** for the following task:

Task: Generate a 5-slide presentation outline for **"Introduction to Large Language Models"** for a university audience.

Critique criteria to use:
1. Does every slide have a clear, specific purpose?
2. Is the progression logical (foundational → advanced)?
3. Does it include at least one interactive/demo slide?
4. Is the content appropriate for a university audience (not too basic, not too advanced)?

Print the outline after each round and note which criteria scores improved.

---
## Summary — Notebook 3 (and Full Course)

| Concept | Key Takeaway |
|---------|-------------|
| **Prompt Chaining** | Break complex tasks into single-purpose steps. Output of A → Input of B. Map-Reduce for bulk processing. Gates for routing. |
| **Meta-Prompting** | Let the LLM write and optimize its own prompts. Evaluate against test cases. Iterate. |
| **Self-Refinement** | Generate → Critique → Revise. 10–25% quality gain per loop. Use self-consistency voting for factual tasks. |

---

## Complete Course Summary

| # | Topic | Notebook | Key Technique |
|---|-------|----------|---------------|
| 1 | Anatomy of a Prompt | NB 1 | 4 components: instruction, context, input, output format |
| 2 | Zero/One/Few-Shot | NB 1 | 3 examples → 94% format compliance |
| 3 | System Prompt Design | NB 1 | Hierarchical: persona → tone → 3–5 rules |
| 4 | Chain-of-Thought | NB 2 | `<reasoning>` tags → +34% accuracy |
| 5 | Structured Output | NB 2 | Pydantic parse() / JSON mode / XML tags |
| 6 | Robustness Testing | NB 2 | Synonym tests + adversarial inputs + positive constraints |
| 7 | Prompt Chaining | NB 3 | Sequential / Map-Reduce / Parallel / Gated |
| 8 | Meta-Prompting | NB 3 | Generate → Critique → Revise loop |

---
*Good prompt engineering is **empirical** — always measure, always test, always iterate.*